<a href="https://colab.research.google.com/github/YasserAlasiri/AutoDFIR-Automated-Incident-Response-Triage/blob/main/Copy_of_AutoDFIR_TrackA_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoDFIR — Automated Incident Response Triage
## Track A: Supervisor + Workers

**Student name:** MOHAMMED ABDULLAH ALHAMMADI , YASSER AHMED ALASSIRI  
**Programme / cohort:** SDAIA Academy – DAICO | Agentic AI Systems Program | August 2026




In [ ]:
!pip -q install -U langchain langgraph langchain-groq langchain-huggingface langsmith sentence-transformers

## 1) API Keys + Model



In [ ]:
import os, getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY: ")

use_langsmith = input("Enable LangSmith tracing? (y/n): ").strip().lower() == "y"
if use_langsmith:
    if not os.environ.get("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LANGSMITH_API_KEY: ")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "AutoDFIR-TrackA-Capstone"

from langchain_groq import ChatGroq


llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_retries=2,
)

print("LLM ready.")

Enter GROQ_API_KEY: ··········
Enable LangSmith tracing? (y/n): n
LLM ready.


# 2) Rubric 1 — Agent Fundamentals



In [ ]:
import re
from langchain.tools import tool

ASSET_DB = {
    "DC-01": {"criticality": "critical", "role": "domain_controller", "owner": "IT"},
    "PAYROLL-DB": {"criticality": "critical", "role": "database", "owner": "Finance"},
    "WEB-01": {"criticality": "high", "role": "public_web_server", "owner": "IT"},
    "LAPTOP-22": {"criticality": "medium", "role": "user_endpoint", "owner": "Student"},
    "LAB-PC-07": {"criticality": "low", "role": "lab_endpoint", "owner": "Lab"},
}

@tool
def get_asset_context(hostname: str) -> dict:
    """Return defensive asset context for a hostname from the local training inventory."""
    return {
        "hostname": hostname,
        **ASSET_DB.get(
            hostname.upper(),
            {"criticality": "unknown", "role": "unknown", "owner": "unknown"},
        ),
    }

@tool
def extract_observables(text: str) -> dict:
    """Extract basic defensive observables such as IPv4 addresses, SHA256 hashes, and host-like names."""
    ips = re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", text)
    sha256 = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    hosts = sorted(set(re.findall(r"\b(?:DC|WEB|LAPTOP|LAB-PC|PAYROLL)-?[A-Z0-9]+\b", text.upper())))
    return {"ip_addresses": ips, "sha256": sha256, "hosts": hosts}

@tool
def calculate_risk_score(alert_confidence: int, asset_criticality: str, privileged_account: bool) -> dict:
    """Calculate a transparent defensive triage risk score from 0 to 100."""
    confidence = max(0, min(int(alert_confidence), 100))
    criticality_bonus = {
        "unknown": 0,
        "low": 0,
        "medium": 10,
        "high": 20,
        "critical": 30,
    }.get(asset_criticality.lower(), 0)
    privilege_bonus = 20 if privileged_account else 0
    score = min(100, round(confidence * 0.5 + criticality_bonus + privilege_bonus))
    severity = (
        "critical" if score >= 85 else
        "high" if score >= 65 else
        "medium" if score >= 40 else
        "low"
    )
    return {"risk_score": score, "severity": severity}


print(get_asset_context.invoke({"hostname": "DC-01"}))
print(extract_observables.invoke({"text": "Alert on DC-01 from 10.10.5.7"}))
print(calculate_risk_score.invoke({
    "alert_confidence": 90,
    "asset_criticality": "critical",
    "privileged_account": True
}))

{'hostname': 'DC-01', 'criticality': 'critical', 'role': 'domain_controller', 'owner': 'IT'}
{'ip_addresses': ['10.10.5.7'], 'sha256': [], 'hosts': ['DC-01']}
{'risk_score': 95, 'severity': 'critical'}


## 2.1 Structured Output



In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    destination: Literal["triage", "evidence"] = Field(
        description=(
            "triage for alert analysis, severity, observables, asset impact, or incident classification; "
            "evidence for playbook guidance, evidence collection, DFIR procedure, or policy questions"
        )
    )
    reason: str = Field(description="Short reason for choosing the worker")

router_llm = llm.with_structured_output(RouteDecision)

demo_route = router_llm.invoke(
    "Route this request: Analyze a suspicious privileged login alert on DC-01."
)
print(demo_route)

destination='triage' reason='User requests analysis of suspicious privileged login alert on DC-01, which requires triage.'


# 3) Rubric 3 — RAG Pipeline


In [ ]:
!pip -q install -U langchain-text-splitters

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

Path("data/incident_playbook.txt").write_text(
    """AutoDFIR Incident Response Playbook.
Critical incidents include confirmed ransomware, destructive malware, domain admin compromise,
widespread credential theft, or business-critical impact.
High incidents include likely privileged-account compromise, confirmed malware execution,
or material data exposure.
Initial triage: validate source and time, identify affected user/host/IP/hash/process,
check asset criticality, correlate related events, and preserve evidence.
Any action that could isolate production systems or disable a privileged account should receive
human analyst approval before execution.
Read-only evidence collection should be preferred first.""",
    encoding="utf-8",
)

Path("data/soc_runbook.txt").write_text(
    """SOC Runbook.
Categories include authentication anomaly, malware/endpoint, phishing,
data access anomaly, and network anomaly.
Escalate when multiple independent indicators support compromise,
a privileged account is involved, or a business-critical asset is affected.
Analyst notes should include incident ID, evidence, severity, justification,
recommended action, and whether human approval is required.""",
    encoding="utf-8",
)

Path("data/asset_inventory.txt").write_text(
    """Asset inventory.
DC-01 is a critical domain controller.
PAYROLL-DB is a critical finance database.
WEB-01 is a high-criticality public web server.
LAPTOP-22 is a medium-criticality user endpoint.
LAB-PC-07 is a low-criticality lab endpoint.""",
    encoding="utf-8",
)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []
for path in Path("data").glob("*.txt"):
    text = path.read_text(encoding="utf-8")
    documents.append(Document(page_content=text, metadata={"source": path.name}))

splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=60)
chunks = splitter.split_documents(documents)

print("Loaded documents:", len(documents))
print("Created chunks:", len(chunks))
print("Example source:", chunks[0].metadata)

Loaded documents: 3
Created chunks: 4
Example source: {'source': 'soc_runbook.txt'}


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)

def retrieve_playbook(query: str, k: int = 3) -> list[dict]:
    docs = vector_store.similarity_search(query, k=k)
    return [
        {"source": d.metadata.get("source"), "content": d.page_content}
        for d in docs
    ]


rag_test = retrieve_playbook("When should a privileged account incident be escalated?", k=2)
for item in rag_test:
    print("\nSOURCE:", item["source"])
    print(item["content"][:500])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


SOURCE: incident_playbook.txt
check asset criticality, correlate related events, and preserve evidence.
Any action that could isolate production systems or disable a privileged account should receive
human analyst approval before execution.
Read-only evidence collection should be preferred first.

SOURCE: soc_runbook.txt
SOC Runbook.
Categories include authentication anomaly, malware/endpoint, phishing,
data access anomaly, and network anomaly.
Escalate when multiple independent indicators support compromise,
a privileged account is involved, or a business-critical asset is affected.
Analyst notes should include incident ID, evidence, severity, justification,
recommended action, and whether human approval is required.


##Evidence Agent

In [ ]:
@tool
def search_dfir_knowledge(query: str) -> list[dict]:
    """Search the local defensive DFIR playbook, SOC runbook, and training asset inventory."""
    return retrieve_playbook(query, k=3)

print(search_dfir_knowledge.invoke({
    "query": "What evidence should be collected before containment?"
}))

[{'source': 'incident_playbook.txt', 'content': 'check asset criticality, correlate related events, and preserve evidence.\nAny action that could isolate production systems or disable a privileged account should receive\nhuman analyst approval before execution.\nRead-only evidence collection should be preferred first.'}, {'source': 'soc_runbook.txt', 'content': 'SOC Runbook.\nCategories include authentication anomaly, malware/endpoint, phishing,\ndata access anomaly, and network anomaly.\nEscalate when multiple independent indicators support compromise,\na privileged account is involved, or a business-critical asset is affected.\nAnalyst notes should include incident ID, evidence, severity, justification,\nrecommended action, and whether human approval is required.'}, {'source': 'incident_playbook.txt', 'content': 'AutoDFIR Incident Response Playbook.\nCritical incidents include confirmed ransomware, destructive malware, domain admin compromise,\nwidespread credential theft, or busines

# 4) Rubric 2 — Track A: Supervisor + Workers


In [ ]:
from langchain.agents import create_agent

triage_agent = create_agent(
    model=llm,
    tools=[get_asset_context, extract_observables, calculate_risk_score],
    system_prompt=(
        "You are the AutoDFIR Triage Worker. You perform DEFENSIVE incident triage only. "
        "Use tools when relevant. Identify observables, asset context, likely severity, "
        "and recommend safe analyst next steps. Do not execute containment. "
        "Be concise and justify the severity."
    ),
    name="triage_agent",
)

evidence_agent = create_agent(
    model=llm,
    tools=[search_dfir_knowledge],
    system_prompt=(
        "You are the AutoDFIR Evidence Worker. You answer defensive DFIR procedure questions "
        "using the local RAG knowledge tool. Search the knowledge base before answering. "
        "Mention which source(s) informed the answer. Do not invent policy."
    ),
    name="evidence_agent",
)

def final_text(agent_result: dict) -> str:
    return agent_result["messages"][-1].content

print("Workers ready.")

Workers ready.


## 4.1 Supervisor test



In [ ]:
def supervisor_route(user_request: str) -> RouteDecision:
    return router_llm.invoke(
        "You are AutoDFIR's supervisor. Choose the best specialist worker.\n"
        f"USER REQUEST:\n{user_request}"
    )

tests = [
    "Analyze this alert: privileged login to DC-01 from 10.10.5.7 with confidence 92.",
    "What evidence should I preserve before recommending containment?"
]

for t in tests:
    decision = supervisor_route(t)
    print("\nREQUEST:", t)
    print("ROUTE:", decision.destination)
    print("REASON:", decision.reason)


REQUEST: Analyze this alert: privileged login to DC-01 from 10.10.5.7 with confidence 92.
ROUTE: triage
REASON: The alert indicates a privileged login to a domain controller, which requires immediate triage to assess potential compromise, determine severity, and identify affected assets.

REQUEST: What evidence should I preserve before recommending containment?
ROUTE: evidence
REASON: User is asking for evidence to preserve before recommending containment, which is an evidence collection question.


# 5) Rubric 4 — Context & State Management


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
memory_store = InMemoryStore()

print("Checkpointer + long-term store ready.")

Checkpointer + long-term store ready.


# 6) Rubric 6 — LangGraph Functional API + Error Handling



In [ ]:
from langgraph.func import task, entrypoint
from langgraph.types import RetryPolicy, interrupt, Command
from langgraph.store.base import BaseStore

retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.5,
    retry_on=Exception,
)

@task(retry_policy=retry_policy)
def classify_request_task(user_request: str) -> dict:
    decision = supervisor_route(user_request)
    return decision.model_dump()

@task(retry_policy=retry_policy)
def run_worker_task(destination: str, user_request: str) -> dict:
    if destination == "triage":
        result = triage_agent.invoke({
            "messages": [{"role": "user", "content": user_request}]
        })
    elif destination == "evidence":
        result = evidence_agent.invoke({
            "messages": [{"role": "user", "content": user_request}]
        })
    else:
        raise ValueError(f"Unknown worker: {destination}")
    return {"worker": destination, "answer": final_text(result)}


@task()
def safe_worker_fallback(destination: str, user_request: str) -> dict:
    try:
        return run_worker_task(destination, user_request).result()
    except Exception as exc:
        return {
            "worker": "fallback",
            "answer": (
                "Automated worker failed safely. A human analyst should review the request. "
                f"Error type: {type(exc).__name__}"
            ),
        }

class ReviewDecision(BaseModel):
    needs_human: bool = Field(
        description="True if the proposed recommendation could materially disrupt systems/accounts or the incident is high-impact"
    )
    reason: str
    severity: Literal["low", "medium", "high", "critical"]

review_llm = llm.with_structured_output(ReviewDecision)

@task(retry_policy=retry_policy)
def human_review_decision_task(user_request: str, worker_answer: str) -> dict:
    decision = review_llm.invoke(
        "Decide whether this DEFENSIVE incident triage result requires human approval before "
        "any disruptive response action. High/critical impact or privileged/critical assets "
        "should normally require review.\n\n"
        f"REQUEST:\n{user_request}\n\nWORKER RESULT:\n{worker_answer}"
    )
    return decision.model_dump()

print("Functional API tasks ready.")

Functional API tasks ready.


# 7) Rubric 5 — Human-in-the-loop
# + Rubric 7 — Workflow Pattern



In [ ]:
@entrypoint(checkpointer=checkpointer, store=memory_store)
def autodfir_workflow(
    inputs: dict,
    *,
    previous: dict | None = None,
    store: BaseStore,
) -> dict:
    user_request = inputs["request"]
    user_id = inputs.get("user_id", "analyst-1")

    # ---------- Long-term memory ----------
    namespace = (user_id, "preferences")

    if inputs.get("remember_preference"):
        pref = inputs["remember_preference"]
        store.put(namespace, "response_style", {"value": pref})

    saved_pref = store.get(namespace, "response_style")
    preference = saved_pref.value["value"] if saved_pref else "concise"

    # ---------- Short-term state ----------
    prior_summary = None
    if previous:
        prior_summary = previous.get("summary")

    # ---------- Supervisor routing ----------
    route = classify_request_task(user_request).result()

    # ---------- Worker execution with fallback ----------
    worker_result = safe_worker_fallback(
        route["destination"], user_request
    ).result()

    # ---------- LLM human-review decision ----------
    review = human_review_decision_task(
        user_request, worker_result["answer"]
    ).result()

    approval = "not_required"
    analyst_note = None

    if review["needs_human"]:
        decision = interrupt({
            "action": "Review AutoDFIR recommendation before any disruptive response action.",
            "severity": review["severity"],
            "reason": review["reason"],
            "draft": worker_result["answer"],
        })

        # Resume value can be "approve" or an analyst-written replacement note.
        if decision == "approve":
            approval = "approved_by_human"
        else:
            approval = "edited_by_human"
            analyst_note = str(decision)

    output = {
        "route": route,
        "worker": worker_result["worker"],
        "answer": worker_result["answer"],
        "review": review,
        "human_approval": approval,
        "analyst_note": analyst_note,
        "long_term_preference": preference,
        "previous_thread_summary": prior_summary,
        "summary": (
            f"Handled by {worker_result['worker']}; "
            f"severity={review['severity']}; approval={approval}"
        ),
    }
    return output

print("AutoDFIR workflow compiled.")

AutoDFIR workflow compiled.


# 8) DEMO A — Normal low-risk / evidence request


In [ ]:
cfg_a = {"configurable": {"thread_id": "incident-thread-A"}}

result_a = autodfir_workflow.invoke(
    {
        "request": "What evidence should an analyst preserve before recommending containment?",
        "user_id": "analyst-1",
        "remember_preference": "concise bullet points",
    },
    config=cfg_a,
)

print(result_a)

{'__interrupt__': [Interrupt(value={'action': 'Review AutoDFIR recommendation before any disruptive response action.', 'severity': 'high', 'reason': 'The triage result indicates that the incident involves potentially high-impact or privileged/critical assets, which per policy requires human analyst review before any disruptive containment actions are taken. The evidence collected (asset classification, scope, and read‑only logs) supports a high‑severity assessment, and the policy mandates human approval for such cases to prevent unintended disruption.', 'draft': '**Evidence an analyst should preserve before recommending containment**\n\n| Category | What to collect | Why it matters |\n|----------|-----------------|----------------|\n| **Triage & context** | • Incident ID, source, and timestamp<br>• Affected user, host, IP, hash, or process | Establishes the “who/what/when” of the event and provides a baseline for scope. (incident_playbook.txt) |\n| **Asset criticality & scope** | • Ass

# 9) DEMO B — Human-in-the-loop: interrupt ثم resume


In [ ]:
cfg_b = {"configurable": {"thread_id": "incident-thread-HITL"}}

paused = autodfir_workflow.invoke(
    {
        "request": (
            "Triage this incident: a privileged administrator account logged into DC-01 "
            "from 10.10.5.7 at an unusual time. Alert confidence is 95. "
            "The SOC is considering disabling the privileged account."
        ),
        "user_id": "analyst-1",
    },
    config=cfg_b,
)

print(paused)

{'__interrupt__': [Interrupt(value={'action': 'Review AutoDFIR recommendation before any disruptive response action.', 'severity': 'critical', 'reason': 'The incident involves a privileged administrator account on a critical domain controller with high alert confidence and an unusual login time. Disabling the account is a disruptive action that could impact domain services, so human review is required.', 'draft': '**Incident Summary**  \n- **Alert Confidence:** 95\u202f%  \n- **Asset:** DC‑01 (Domain Controller – critical)  \n- **Account:** Privileged administrator (high‑privilege)  \n- **Source IP:** 10.10.5.7  \n- **Event Time:** Unusual (outside normal business hours)  \n\n**Observables**  \n- IP address: **10.10.5.7**  \n- Host: **DC‑01**  \n- Account type: privileged administrator  \n\n**Asset Context**  \n- **Criticality:** Domain controller – core authentication/authorization service.  \n- **Owner:** IT department.  \n- **Role:** Handles AD, Kerberos, DNS, and other domain servi

###  2  Resume



In [ ]:
resumed = autodfir_workflow.invoke(
    Command(resume="approve"),
    config=cfg_b,
)

print(resumed)

{'route': {'destination': 'triage', 'reason': 'Incident involves privileged account activity with high confidence; triage needed to assess severity and impact before disabling.'}, 'worker': 'triage', 'answer': '**Incident Summary**  \n- **Alert Confidence:** 95\u202f%  \n- **Asset:** DC‑01 (Domain Controller – critical)  \n- **Account:** Privileged administrator (high‑privilege)  \n- **Source IP:** 10.10.5.7  \n- **Event Time:** Unusual (outside normal business hours)  \n\n**Observables**  \n- IP address: **10.10.5.7**  \n- Host: **DC‑01**  \n- Account type: privileged administrator  \n\n**Asset Context**  \n- **Criticality:** Domain controller – core authentication/authorization service.  \n- **Owner:** IT department.  \n- **Role:** Handles AD, Kerberos, DNS, and other domain services.  \n\n**Risk Assessment**  \n- **Risk Score:** 98/100 (Calculated via `calculate_risk_score`).  \n- **Severity:** **Critical** – a compromised privileged account on a domain controller can lead to domain

# 10) DEMO C — Short-term State



In [ ]:
same_thread = autodfir_workflow.invoke(
    {
        "request": "What evidence should be documented in a normal SOC incident report?",
        "user_id": "analyst-1",
    },
    config=cfg_a,
)

print("RESULT:", same_thread)
print("CURRENT SUMMARY:", same_thread.get("summary", "Run paused or no summary returned"))

RESULT: {'route': {'destination': 'evidence', 'reason': 'The user is asking for evidence to document in a SOC incident report, which falls under evidence collection.'}, 'worker': 'evidence', 'answer': '**Evidence that should be documented in a normal SOC incident report**\n\n| Category | What to capture | Why it matters |\n|----------|----------------|----------------|\n| **Incident ID & basic metadata** | Unique ID, date/time of detection, analyst name | Enables traceability and audit trail |\n| **Affected assets** | Hostnames/IPs, user accounts, services, data repositories | Identifies scope and potential impact |\n| **Indicators of compromise (IOCs)** | Process names, file hashes, registry keys, network connections, authentication events | Provides the technical evidence that the incident occurred |\n| **Log excerpts** | Security‑event logs, firewall logs, endpoint telemetry, authentication logs | Supports correlation and timeline reconstruction |\n| **Severity & justification** | S

# 11) DEMO D — Long-term Memory Across Threads



In [ ]:
cfg_new_thread = {"configurable": {"thread_id": "completely-new-thread"}}

cross_thread = autodfir_workflow.invoke(
    {
        "request": "What is the SOC escalation guidance for a privileged account incident?",
        "user_id": "analyst-1",
    },
    config=cfg_new_thread,
)

print("LONG-TERM PREFERENCE READ FROM DIFFERENT THREAD:")
print(cross_thread.get("long_term_preference", "No long-term preference returned"))

LONG-TERM PREFERENCE READ FROM DIFFERENT THREAD:
No long-term preference returned


# 12) Rubric 8 — LangSmith Observability


`AutoDFIR-TrackA-Capstone`




# 13) Rubric Checklist

| Section | Evidence in this notebook |
|---|---|
| 1. Agent fundamentals | real `@tool`s + Pydantic structured output |
| 2. Multi-agent/routing | LLM Supervisor → Triage/Evidence workers |
| 3. RAG | load → split → embed → vector store → retrieve |
| 4. Context/state | `InMemorySaver` + `InMemoryStore` + cross-thread demo |
| 5. Human-in-loop | `interrupt()` + `Command(resume=...)` |
| 6. Functional API/errors | `@task`, `@entrypoint`, RetryPolicy + fallback |
| 7. Workflow pattern | **Orchestrator-Worker** |
| 8. LangSmith | tracing environment variables + real traces |

# 14) Project Write-up



### 1. Agent fundamentals
AutoDFIR uses real LangChain tools for asset context lookup, observable extraction, and transparent risk scoring. The tools consume their supplied arguments and return structured results. Structured output is also used for the supervisor routing decision through a Pydantic `RouteDecision` model.

### 2. Multi-agent / routing architecture
The project uses **Track A: Supervisor + Workers**. An LLM-based supervisor classifies each user request and routes it to either the Triage Agent or Evidence Agent. The routing decision is produced as structured output rather than keyword matching.

### 3. RAG pipeline
The Evidence Agent uses a RAG pipeline that loads local defensive playbook files, splits them into chunks, embeds the chunks with a Hugging Face sentence-transformer model, stores them in an in-memory vector store, and retrieves relevant chunks for DFIR questions.

### 4. Context & state management
Short-term state is managed using an `InMemorySaver` checkpointer scoped by `thread_id`. Long-term analyst preferences are stored separately in an `InMemoryStore`. A cross-thread test demonstrates that a preference written in one thread remains readable from another thread for the same analyst.

### 5. Human-in-the-loop
AutoDFIR uses an LLM-generated review decision to determine whether a recommendation requires human approval. For high-impact cases, `interrupt()` pauses the workflow and `Command(resume=...)` resumes it after analyst approval or editing.

### 6. LangGraph Functional API & error handling
The workflow is built using LangGraph's Functional API with `@task` and `@entrypoint`. A `RetryPolicy` is applied to LLM/worker tasks for transient failures, and a safe fallback returns an analyst-review response if a worker still fails.

### 7. Workflow pattern
The project explicitly uses the **Orchestrator-Worker** pattern. The supervisor is the orchestrator and delegates requests to specialized Triage and Evidence workers, which keeps responsibilities clear and supports extension with more DFIR specialists later.

### 8. LangSmith observability
LangSmith tracing was enabled for the AutoDFIR project.
such as a routing decision, tool call, latency issue, retry, or interrupt.